# [반도체장비예지보전실무] AGV/OHT 장비 상태 분류 최종 프로젝트 실습
본 노트북은 **미션 01(데이터 탐색) ~ 미션 04(최종 모델 검증 및 도입 권고)**의 핵심 실습 코드와 실행 결과를 하나로 통합한 주피터 노트북입니다.
- **대상 장비:** AGV (무인운반차) 중심 분석 (미션 01에서 OHT와 비교 포함)
- **최종 선택 센서 (5개):** `NTC_temp`, `CT1_current`, `IR_temp_max`, `PM10`, `CT2_current`
- **최종 선택 알고리즘:** `RandomForest` (랜덤포레스트 분류기)
- **상태 레이블:** 정상(0), 관심(1), 경고(2), 위험(3)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score, recall_score, confusion_matrix

# 데이터 불러오기 (CSV 파일 4개가 노트북과 같은 폴더에 있어야 합니다)
train_agv = pd.read_csv("Training_AGV.csv", parse_dates=["timestamp"])
val_agv = pd.read_csv("Validation_AGV.csv", parse_dates=["timestamp"])
train_oht = pd.read_csv("Training_OHT.csv", parse_dates=["timestamp"])
val_oht = pd.read_csv("Validation_OHT.csv", parse_dates=["timestamp"])

print("AGV Training 데이터 크기:", train_agv.shape)
print("AGV Validation 데이터 크기:", val_agv.shape)
print("OHT Training 데이터 크기:", train_oht.shape)
print("OHT Validation 데이터 크기:", val_oht.shape)

---
## MISSION 01. 두 장비(AGV vs OHT)의 센서 데이터 탐색
1. **장비 상태(state) 분포 확인:** 정상(0), 관심(1), 경고(2), 위험(3)의 데이터 수와 비율을 확인합니다.
2. **센서와 장비 상태(state)의 상관관계 비교:** 어떤 센서가 장비 열화(상태 숫자 증가)와 함께 뚜렷하게 변하는지 확인합니다.

In [ ]:
# 1. AGV 및 OHT 상태별 데이터 분포 확인
for name, df in [("AGV Training (01~16번)", train_agv), ("OHT Training (01~16번)", train_oht)]:
    state_counts = df["state"].value_counts().reindex([0, 1, 2, 3], fill_value=0)
    state_summary = pd.DataFrame({
        "데이터 수": state_counts,
        "비율(%)": (state_counts / len(df) * 100).round(1)
    })
    print(f"=== {name} 상태 분포 ===")
    print(state_summary)
    print()

In [ ]:
# 2. 초기 5대 장비 기준 센서와 상태(state) 간의 상관계수 비교
SENSOR_ALL = [
    "PM10", "PM2.5", "PM1.0", "NTC_temp",
    "CT1_current", "CT2_current", "CT3_current", "CT4_current",
    "IR_temp_max", "ex_temp", "ex_humidity"
]

agv_5 = train_agv[train_agv["device_id"].isin([f"agv0{i}" for i in range(1, 6)])]
oht_5 = train_oht[train_oht["device_id"].isin([f"oht0{i}" for i in range(1, 6)])]

corr_comp = pd.DataFrame({
    "AGV_state_corr": agv_5[SENSOR_ALL + ["state"]].corr()["state"].drop("state").round(4),
    "OHT_state_corr": oht_5[SENSOR_ALL + ["state"]].corr()["state"].drop("state").round(4)
}).sort_values(by="AGV_state_corr", key=abs, ascending=False)

print("=== AGV vs OHT 센서-상태 상관계수 비교 (01~05번 시범 장비) ===")
print(corr_comp)

---
## MISSION 02. 적은 센서로 장비 상태 분류하기 (특성 선택 및 모델 S 생성)
- **실험 방법:** 시범 장비 5대 중 `agv01~03`으로 학습하고, 센서 조합과 알고리즘(`LogisticRegression` vs `RandomForest`)을 비교합니다.
- **최종 선정 센서 조합 (5개):** `["NTC_temp", "CT1_current", "IR_temp_max", "PM10", "CT2_current"]`
  - 온도 센서 2종(`NTC_temp`, `IR_temp_max`), 주요 전류 센서 2종(`CT1_current`, `CT2_current`), 마모 분진 센서 1종(`PM10`)을 결합하여 11개 전체 센서와 대등한 분류 성능을 확보하면서 관리 센서를 절반 이하로 축소했습니다.

In [ ]:
# 모델 생성 및 평가 공통 함수 정의
def make_model(algorithm):
    if algorithm == "LogisticRegression":
        return Pipeline([
            ("scaler", StandardScaler()),
            ("classifier", LogisticRegression(
                max_iter=1500,
                class_weight="balanced",
                random_state=42
            ))
        ])
    if algorithm == "RandomForest":
        return RandomForestClassifier(
            n_estimators=100,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        )
    raise ValueError(f"알 수 없는 알고리즘: {algorithm}")

def evaluate_model(model, X, y_true):
    y_pred = model.predict(X)
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Macro F1": f1_score(y_true, y_pred, labels=[0, 1, 2, 3], average="macro", zero_division=0),
        "Danger Recall": recall_score(y_true, y_pred, labels=[3], average="macro", zero_division=0)
    }

# 미션 02 소량 학습 데이터 분할 (agv01~03 학습, agv04 검증, agv05 평가)
m2_train = train_agv[train_agv["device_id"].isin(["agv01", "agv02", "agv03"])].copy()
m2_val = train_agv[train_agv["device_id"].isin(["agv04"])].copy()
m2_test = train_agv[train_agv["device_id"].isin(["agv05"])].copy()

# 선택한 핵심 5개 센서와 랜덤포레스트로 모델 S 학습 및 저장
features = ["NTC_temp", "CT1_current", "IR_temp_max", "PM10", "CT2_current"]
DEVICE = "AGV"
ALGORITHM = "RandomForest"

model_S = make_model(ALGORITHM)
model_S.fit(m2_train[features], m2_train["state"])

# 모델 S 패키지 저장
Path("saved_models").mkdir(exist_ok=True)
model_s_path = Path("saved_models/AGV_RandomForest_5_sensors.joblib")
joblib.dump({
    "model": model_S,
    "features": features,
    "device": DEVICE,
    "algorithm": ALGORITHM
}, model_s_path)

print("미션 02 모델 S 저장 완료:", model_s_path)
print("agv04 검증 성능:", pd.DataFrame([evaluate_model(model_S, m2_val[features], m2_val["state"])]).round(4).to_dict(orient="records")[0])
print("agv05 평가 성능:", pd.DataFrame([evaluate_model(model_S, m2_test[features], m2_test["state"])]).round(4).to_dict(orient="records")[0])

---
## MISSION 03. 소량 데이터 모델(S) vs 대규모 데이터 모델(L) 비교 검증
- **실험 조건:**
  - **모델 S (Small):** `agv01~03` (3대, 6,498행)으로 학습한 기존 모델
  - **모델 L (Large):** `agv01~12` (12대, 28,879행)으로 새롭게 재학습한 대규모 모델
  - **비교 검증 장비:** `agv13~16` (4대, 10,104행)

In [ ]:
# A-4. 장비 ID를 기준으로 학습(01~12번) 및 검증(13~16번) 데이터 분리
large_data = train_agv.copy()
prefix = DEVICE.lower()
train_ids = [f"{prefix}{i:02d}" for i in range(1, 13)]
val_ids = [f"{prefix}{i:02d}" for i in range(13, 17)]

train_df = large_data[large_data["device_id"].isin(train_ids)].copy()
val_df = large_data[large_data["device_id"].isin(val_ids)].copy()

print("대규모 학습 데이터(01~12번):", train_df.shape)
print("검증 데이터(13~16번):", val_df.shape)

# B-2. [직접 작성 01] 모델 S로 13~16번 검증 장비 예측 및 평가
X_val = val_df[features]
y_val = val_df["state"]
y_pred_S = model_S.predict(X_val)
scores_S = evaluate_model(model_S, X_val, y_val)

# C-4. 모델 L(01~12번 학습) 생성, 학습 및 평가
model_L = make_model(ALGORITHM)
X_train = train_df[features]
y_train = train_df["state"]
model_L.fit(X_train, y_train)
scores_L = evaluate_model(model_L, X_val, y_val)

# D-1. 모델 S와 모델 L의 전체 성능 비교
comparison_df = pd.DataFrame([
    {"Model": "S (Small, 01~03번 학습)", **scores_S},
    {"Model": "L (Large, 01~12번 학습)", **scores_L}
])
print("\n[모델 S vs 모델 L 검증 성능 비교 (agv13~16)]")
print(comparison_df.round(4).to_string(index=False))

In [ ]:
# D-2. [직접 작성 02] 장비별(agv13~16) 성능 편차 계산하기
def evaluate_by_device(model, df, features):
    results = []
    for device_id in sorted(df["device_id"].unique()):
        part = df[df["device_id"] == device_id]
        X = part[features]
        y = part["state"]
        scores = evaluate_model(model, X, y)
        results.append({
            "device_id": device_id,
            "Macro F1": scores["Macro F1"],
            "Danger Recall": scores["Danger Recall"]
        })
    return pd.DataFrame(results)

device_scores_S = evaluate_by_device(model_S, val_df, features)
device_scores_L = evaluate_by_device(model_L, val_df, features)

device_comparison = device_scores_S.merge(
    device_scores_L,
    on="device_id",
    suffixes=("_S", "_L")
)
print("[검증 장비별(agv13~16) 성능 비교]")
print(device_comparison.round(4).to_string(index=False))

---
## MISSION 04. 최종 모델 학습(REFIT) 및 신규 장비(17~18번) 최종 평가
- **최종 학습 방식 (`REFIT`):** 검증이 완료된 `01~16번` 장비 전체 데이터(38,983행)로 최종 모델을 재학습합니다.
- **독립 최종 평가:** 모델 선택 과정에 한 번도 사용하지 않은 `agv17~18` (2대, 5,052행) 데이터로 최종 일반화 성능과 오분류 행렬(Confusion Matrix)을 확인합니다.

In [ ]:
# C-3. [직접 작성 01] 01~16번 전체 데이터로 최종 모델(REFIT) 학습 및 저장
FINAL_FEATURES = list(features)
FINAL_ALGORITHM = ALGORITHM
FINAL_METHOD = "REFIT"

X_final_train = large_data[FINAL_FEATURES]
y_final_train = large_data["state"]

final_model = make_model(FINAL_ALGORITHM)
final_model.fit(X_final_train, y_final_train)

final_model_path = Path("saved_models/AGV_final_model.joblib")
joblib.dump({
    "model": final_model,
    "features": FINAL_FEATURES,
    "device": DEVICE,
    "algorithm": FINAL_ALGORITHM,
    "training_method": FINAL_METHOD,
    "target_labels": [0, 1, 2, 3]
}, final_model_path)

print("최종 학습 데이터 크기:", X_final_train.shape)
print("최종 모델 저장 완료:", final_model_path)

# D-3. [직접 작성 02] 신규 장비(agv17~18) 최종 성능 평가
final_test_df = val_agv.copy()
X_final_test = final_test_df[FINAL_FEATURES]
y_final_test = final_test_df["state"]

final_scores = evaluate_model(final_model, X_final_test, y_final_test)
print("\n[신규 장비(agv17~18) 최종 평가 결과]")
print(pd.DataFrame([final_scores]).round(4).to_string(index=False))

# D-5. 오분류 행렬(Confusion Matrix) 확인
y_final_pred = final_model.predict(X_final_test)
cm = confusion_matrix(y_final_test, y_final_pred, labels=[0, 1, 2, 3])
cm_df = pd.DataFrame(
    cm,
    index=["실제 정상(0)", "실제 관심(1)", "실제 경고(2)", "실제 위험(3)"],
    columns=["예측 정상(0)", "예측 관심(1)", "예측 경고(2)", "예측 위험(3)"]
)
print("\n[신규 장비(agv17~18) 오분류 행렬(Confusion Matrix)]")
print(cm_df)